# Pipeline de Generación del Dataset de Evaluación (sample_1000.csv)

Este notebook documenta paso a paso el armado transparente y reproducible de la muestra de evaluación acordada (**REQ-07**):
1. **Descarga:** Descarga automática del dataset oficial de NVD / HuggingFace (`CVE_CWE_2025.csv`, ~104 MB, ~280k CVEs) si no existe localmente.
2. **Carga en Pandas:** Lectura directa del CSV con `pd.read_csv()`.
3. **Filtros Temporales y de Limpieza:**
   - Filtrar CVEs del **segundo semestre de 2024** (`CVE-2024-39...` a `CVE-2024-5...`).
   - Eliminar nulos en `DESCRIPTION` o `SEVERITY` (`dropna`).
   - Eliminar descripciones duplicadas (`drop_duplicates`).
4. **Muestreo Estratificado:** Selección determinista (`random_state=42`) para cumplir exactamente con la distribución poblacional:
   - `MEDIUM`: 450 (45.0%)
   - `HIGH`: 370 (37.0%)
   - `CRITICAL`: 164 (16.4%)
   - `LOW`: 16 (1.6%)
5. **Exportación:** Guardado de `sample_1000.csv`.

## 1. Descarga del archivo fuente (si no existe)

In [ ]:
import os
import urllib.request
import pandas as pd

DATASET_URL = "https://huggingface.co/datasets/stasvinokur/cve-and-cwe-dataset-1999-2025/resolve/main/CVE_CWE_2025.csv"
RAW_CSV_PATH = "CVE_CWE_2025.csv"

if not os.path.exists(RAW_CSV_PATH):
    print(f"Descargando dataset desde {DATASET_URL}...")
    urllib.request.urlretrieve(DATASET_URL, RAW_CSV_PATH)
    print("Descarga completada con exito.")
else:
    print(f"El archivo {RAW_CSV_PATH} ya existe localmente.")

## 2. Carga directa con Pandas

In [ ]:
print("Cargando dataset completo en memoria...")
df = pd.read_csv(RAW_CSV_PATH, low_memory=False)
print(f"Total de filas cargadas: {len(df):,}")
df.head()

## 3. Limpieza y Filtros
- **Partición temporal estricta:** Tomamos CVEs del año 2024 con numeración mayor a 30.000 (segundo semestre de 2024, posteriores al knowledge cutoff de los modelos).
- **Descarte de nulos:** Filas que tengan nulo en `DESCRIPTION` o `SEVERITY`.
- **Descarte de duplicados:** Registros con descripciones idénticas.

In [ ]:
# 1. Filtrar solo CVEs del 2do semestre de 2024
cve_str = df["CVE-ID"].astype(str)
df_2024_h2 = df[cve_str.str.contains(r"^CVE-2024-(3[9-9]|4[0-9]|5[0-9])", regex=True)].copy()
print(f"CVEs del 2do semestre 2024 encontrados: {len(df_2024_h2):,}")

# 2. Limpieza de nulos
valid_classes = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}
df_clean = df_2024_h2[df_2024_h2["SEVERITY"].isin(valid_classes)].copy()
df_clean = df_clean.dropna(subset=["DESCRIPTION", "SEVERITY"])

# 3. Descarte de duplicados de descripcion
total_antes = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["DESCRIPTION"])
print(f"Filas antes de deduplicar: {total_antes} -> Despues: {len(df_clean)}")

print("\nDistribucion poblacional en 2024-H2 (% del total):")
pcts = (df_clean["SEVERITY"].value_counts(normalize=True) * 100).round(2)
for sev, val in pcts.items():
    print(f"{sev:<10}: {val:>5.2f} %")

## 4. Muestreo Estratificado (Criterios exactos de la propuesta)
Seleccionamos al azar (`random_state=42`) los 1.000 casos respetando los porcentajes acordados con el docente:
- **MEDIUM:** 450 (45.0%)
- **HIGH:** 370 (37.0%)
- **CRITICAL:** 164 (16.4%)
- **LOW:** 16 (1.6%)

In [ ]:
TARGETS = {
    "MEDIUM": 450,
    "HIGH": 370,
    "CRITICAL": 164,
    "LOW": 16,
}

RANDOM_SEED = 42
parts = []
for sev, n in TARGETS.items():
    subset = df_clean[df_clean["SEVERITY"] == sev]
    parts.append(subset.sample(n=n, random_state=RANDOM_SEED))

# Combinar y barajar
sample_1000 = pd.concat(parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

# Conservar unicamente las columnas indispensables: CVE-ID, SEVERITY y DESCRIPTION
cols = ["CVE-ID", "SEVERITY", "DESCRIPTION"]
sample_1000 = sample_1000[cols]

print(f"Total de casos en la muestra: {len(sample_1000)}")
print("\nConteo exacto por severidad:")
print(sample_1000["SEVERITY"].value_counts())
sample_1000.head()

## 5. Exportación del archivo final sample_1000.csv

In [ ]:
sample_1000.to_csv("sample_1000.csv", index=False)
print("sample_1000.csv guardado exitosamente.")